## Analyze misclassified examples and their characteristics by manually clustering error cases, detecting anomalies in feature distributions, and performing root-cause analysis to determine if specific subgroups are disproportionately affected.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix, accuracy_score

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
df = pd.read_csv("heart.csv")
df = df.drop_duplicates().copy()
df = df.dropna().copy()

if "target" not in df.columns:
    raise ValueError("Dataset must contain a target column.")

X = df.drop(columns=["target"])
y = df["target"]

if y.nunique() != 2 or set(y.unique()) != {0, 1}:
    raise ValueError("Target must contain binary values 0 and 1.")

X = pd.get_dummies(X, drop_first=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

print("Dataset shape:", df.shape)
print("Test records:", len(X_test))
print("Accuracy:", round(accuracy_score(y_test, predictions), 4))

Dataset shape: (302, 14)
Test records: 61
Accuracy: 0.7869


In [3]:
errors = X_test.copy()

errors["Actual"] = y_test
errors["Predicted"] = predictions
errors["Probability"] = probabilities

errors["Error Type"] = np.select(
    [
        (errors["Actual"] == 0) & (errors["Predicted"] == 1),
        (errors["Actual"] == 1) & (errors["Predicted"] == 0)
    ],
    ["False Positive", "False Negative"],
    default="Correct"
)

errors = errors[errors["Error Type"] != "Correct"].copy()

print("Total misclassified records:", len(errors))
print("False Positives:", (errors["Error Type"] == "False Positive").sum())
print("False Negatives:", (errors["Error Type"] == "False Negative").sum())

display(errors.head(10))

Total misclassified records: 13
False Positives: 10
False Negatives: 3


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,Actual,Predicted,Probability,Error Type
194,60,1,2,140,185,0,0,155,0,3.0,1,0,2,0,1,0.65,False Positive
188,50,1,2,140,233,0,1,163,0,0.6,1,1,3,0,1,0.68,False Positive
228,59,1,3,170,288,0,0,159,0,0.2,1,0,3,0,1,0.69,False Positive
95,53,1,0,142,226,0,0,111,1,0.0,2,0,3,1,0,0.32,False Negative
267,49,1,2,118,149,0,0,126,0,0.8,2,3,2,0,1,0.78,False Positive
302,57,0,1,130,236,0,0,174,0,0.0,1,1,2,0,1,0.73,False Positive
150,66,1,0,160,228,0,0,138,0,2.3,2,0,1,1,0,0.48,False Negative
70,54,1,2,120,258,0,0,147,0,0.4,1,0,3,1,0,0.44,False Negative
281,52,1,0,128,204,1,1,156,1,1.0,1,0,0,0,1,0.67,False Positive
272,67,1,0,120,237,0,1,71,0,1.0,1,0,2,0,1,0.65,False Positive


In [4]:
errors = X_test.copy()

errors["Actual"] = y_test
errors["Predicted"] = predictions
errors["Probability"] = probabilities

errors["Error Type"] = np.select(
    [
        (errors["Actual"] == 0) & (errors["Predicted"] == 1),
        (errors["Actual"] == 1) & (errors["Predicted"] == 0)
    ],
    ["False Positive", "False Negative"],
    default="Correct"
)

errors = errors[errors["Error Type"] != "Correct"].copy()

print("Total misclassified records:", len(errors))
print("False Positives:", (errors["Error Type"] == "False Positive").sum())
print("False Negatives:", (errors["Error Type"] == "False Negative").sum())

display(errors.head(10))

Total misclassified records: 13
False Positives: 10
False Negatives: 3


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,Actual,Predicted,Probability,Error Type
194,60,1,2,140,185,0,0,155,0,3.0,1,0,2,0,1,0.65,False Positive
188,50,1,2,140,233,0,1,163,0,0.6,1,1,3,0,1,0.68,False Positive
228,59,1,3,170,288,0,0,159,0,0.2,1,0,3,0,1,0.69,False Positive
95,53,1,0,142,226,0,0,111,1,0.0,2,0,3,1,0,0.32,False Negative
267,49,1,2,118,149,0,0,126,0,0.8,2,3,2,0,1,0.78,False Positive
302,57,0,1,130,236,0,0,174,0,0.0,1,1,2,0,1,0.73,False Positive
150,66,1,0,160,228,0,0,138,0,2.3,2,0,1,1,0,0.48,False Negative
70,54,1,2,120,258,0,0,147,0,0.4,1,0,3,1,0,0.44,False Negative
281,52,1,0,128,204,1,1,156,1,1.0,1,0,0,0,1,0.67,False Positive
272,67,1,0,120,237,0,1,71,0,1.0,1,0,2,0,1,0.65,False Positive


In [5]:
feature_columns = X.columns.tolist()

if len(errors) >= 4:
    scaler = StandardScaler()

    scaled_errors = scaler.fit_transform(
        errors[feature_columns]
    )

    number_of_clusters = min(
        3, len(errors) // 2
    )

    clustering = KMeans(
        n_clusters=number_of_clusters,
        random_state=42,
        n_init=10
    )

    errors["Error Cluster"] = clustering.fit_predict(
        scaled_errors
    )

    print("Error clustering completed")

    cluster_summary = errors.groupby("Error Cluster").agg(
        Error_Count=("Error Type", "size"),
        False_Positives=(
            "Error Type",
            lambda s: (s == "False Positive").sum()
        ),
        False_Negatives=(
            "Error Type",
            lambda s: (s == "False Negative").sum()
        ),
        Average_Probability=("Probability", "mean")
    )

    display(cluster_summary.round(3))

else:
    errors["Error Cluster"] = -1
    print("Too few errors for reliable clustering.")

Error clustering completed


D:\Softwares\Anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


,Error_Count,False_Positives,False_Negatives,Average_Probability
Error Cluster,,,,
0,7,5,2,0.621
1,1,1,0,0.670
2,5,4,1,0.630


In [6]:
Q1 = X_train.quantile(0.25)
Q3 = X_train.quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outlier_flags = (
    (X_test < lower_limit) |
    (X_test > upper_limit)
)

anomaly_count = outlier_flags.sum(axis=1)

errors["Unusual Feature Count"] = anomaly_count.loc[errors.index]

errors["Anomaly Flag"] = np.where(
    errors["Unusual Feature Count"] > 0,
    "Potential Anomaly",
    "No IQR Anomaly"
)

print("Potential anomaly counts in error cases:")
display(errors["Anomaly Flag"].value_counts().to_frame("Records"))

print("Error records with the most unusual feature values:")
display(
    errors.sort_values(
        "Unusual Feature Count", ascending=False
    ).head(10)
)

Potential anomaly counts in error cases:


,Records
Anomaly Flag,
No IQR Anomaly,10
Potential Anomaly,3


Error records with the most unusual feature values:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,Actual,Predicted,Probability,Error Type,Error Cluster,Unusual Feature Count,Anomaly Flag
281,52,1,0,128,204,1,1,156,1,1.0,1,0,0,0,1,0.67,False Positive,1,2,Potential Anomaly
267,49,1,2,118,149,0,0,126,0,0.8,2,3,2,0,1,0.78,False Positive,2,1,Potential Anomaly
272,67,1,0,120,237,0,1,71,0,1.0,1,0,2,0,1,0.65,False Positive,0,1,Potential Anomaly
194,60,1,2,140,185,0,0,155,0,3.0,1,0,2,0,1,0.65,False Positive,2,0,No IQR Anomaly
188,50,1,2,140,233,0,1,163,0,0.6,1,1,3,0,1,0.68,False Positive,0,0,No IQR Anomaly
228,59,1,3,170,288,0,0,159,0,0.2,1,0,3,0,1,0.69,False Positive,0,0,No IQR Anomaly
95,53,1,0,142,226,0,0,111,1,0.0,2,0,3,1,0,0.32,False Negative,0,0,No IQR Anomaly
302,57,0,1,130,236,0,0,174,0,0.0,1,1,2,0,1,0.73,False Positive,0,0,No IQR Anomaly
150,66,1,0,160,228,0,0,138,0,2.3,2,0,1,1,0,0.48,False Negative,2,0,No IQR Anomaly
70,54,1,2,120,258,0,0,147,0,0.4,1,0,3,1,0,0.44,False Negative,0,0,No IQR Anomaly


In [7]:
raw_test = df.loc[X_test.index].copy()

raw_test["Actual"] = y_test
raw_test["Predicted"] = predictions
raw_test["Error"] = (
    raw_test["Actual"] != raw_test["Predicted"]
).astype(int)

if "age" in raw_test.columns:
    raw_test["Age Group"] = pd.cut(
        raw_test["age"],
        bins=[0, 40, 50, 60, 120],
        labels=["Under 40", "40-49", "50-59", "60+"],
        right=False
    )

subgroup_columns = [
    col for col in [
        "Age Group", "sex", "cp", "exang", "fbs"
    ]
    if col in raw_test.columns
]

subgroup_results = []

for column in subgroup_columns:
    grouped = raw_test.groupby(
        column, observed=True
    )["Error"].agg(["count", "sum", "mean"])

    for group_value, row in grouped.iterrows():
        subgroup_results.append({
            "Feature": column,
            "Group": str(group_value),
            "Records": int(row["count"]),
            "Errors": int(row["sum"]),
            "Error Rate": row["mean"]
        })

subgroup_table = pd.DataFrame(subgroup_results)

if not subgroup_table.empty:
    subgroup_table = subgroup_table.sort_values(
        "Error Rate", ascending=False
    )

    display(subgroup_table.round(3))
else:
    print("No supported subgroup columns were found.")

,Feature,Group,Records,Errors,Error Rate
9,cp,3,8,3,0.375
8,cp,2,17,5,0.294
5,sex,1,41,12,0.293
3,Age Group,60+,18,5,0.278
2,Age Group,50-59,27,7,0.259
10,exang,0,44,11,0.250
12,fbs,0,50,12,0.240
7,cp,1,7,1,0.143
6,cp,0,29,4,0.138
11,exang,1,17,2,0.118


In [8]:
comparison_data = X_test.copy()

comparison_data["Error Type"] = np.select(
    [
        (y_test == 0) & (predictions == 1),
        (y_test == 1) & (predictions == 0)
    ],
    ["False Positive", "False Negative"],
    default="Correct"
)

fp_records = comparison_data[
    comparison_data["Error Type"] == "False Positive"
]

fn_records = comparison_data[
    comparison_data["Error Type"] == "False Negative"
]

correct_records = comparison_data[
    comparison_data["Error Type"] == "Correct"
]

root_cause_table = pd.DataFrame({
    "False Positive Mean": fp_records[feature_columns].mean(),
    "False Negative Mean": fn_records[feature_columns].mean(),
    "Correct Prediction Mean": correct_records[feature_columns].mean()
})

display(root_cause_table.round(3))

,False Positive Mean,False Negative Mean,Correct Prediction Mean
age,58.10,57.667,53.854
sex,0.90,1.000,0.604
cp,1.80,0.667,0.938
trestbps,136.60,140.667,130.771
chol,218.20,237.333,264.021
fbs,0.10,0.000,0.208
restecg,0.50,0.000,0.438
thalach,146.10,132.000,148.875
exang,0.10,0.333,0.312
oldpeak,1.08,0.900,0.904


In [9]:
total_errors = len(errors)
total_test = len(X_test)

fp_count = int((predictions[y_test.index.get_indexer(y_test.index)] != y_test).sum()) if False else int(
    ((y_test == 0) & (predictions == 1)).sum()
)

fn_count = int(
    ((y_test == 1) & (predictions == 0)).sum()
)

if total_test > 0:
    overall_error_rate = total_errors / total_test
else:
    overall_error_rate = 0

if not subgroup_table.empty:
    largest_subgroup_errors = subgroup_table.head(5).to_string(index=False)
else:
    largest_subgroup_errors = "No subgroup results available."

if total_errors > 0:
    cluster_counts = errors["Error Cluster"].value_counts().to_dict()
else:
    cluster_counts = {}

report = f"""
MISCLASSIFICATION AND ROOT-CAUSE ANALYSIS
Heart Disease Classification

1. OBJECTIVE

To examine false positives and false negatives, cluster similar error
cases, detect unusual feature values, and investigate subgroup error rates.

2. MODEL RESULTS

Test records: {total_test}
Misclassified records: {total_errors}
False positives: {fp_count}
False negatives: {fn_count}
Overall error rate: {overall_error_rate:.4f}
Accuracy: {accuracy_score(y_test, predictions):.4f}

3. ERROR CLUSTERING

Cluster counts: {cluster_counts}

Clusters represent groups of records with similar standardized
feature patterns. They do not represent clinical diagnoses.

4. ANOMALY ANALYSIS

Potential anomalies were flagged using the IQR rule.
Unusual values require verification and should not automatically
be treated as data errors.

5. SUBGROUP ANALYSIS

{largest_subgroup_errors}

Subgroup error rates can be unstable when group sample sizes are small.
Differences suggest areas for further investigation, not proof of bias.

6. ROOT-CAUSE FINDINGS

Feature means were compared across false positives, false negatives,
and correct predictions. Differences may suggest model weaknesses,
overlapping feature patterns, or data quality issues.

These observational comparisons do not prove causal explanations.

7. PROPOSED IMPROVEMENTS

- Review data quality and feature encoding.
- Inspect clusters containing repeated error patterns.
- Validate subgroup findings on a larger independent dataset.
- Check calibration and decision thresholds on validation data.
- Evaluate performance across relevant demographic subgroups.
- Obtain domain-expert review before clinical interpretation.

8. CONCLUSION

This analysis identifies patterns in model errors and highlights
subgroups that may need further evaluation. Findings should be
validated on additional data before changing the model or using
it in real-world clinical decisions.
"""

with open(
    "Heart_Disease_Misclassification_Root_Cause_Report.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(report)

print(report)
print("Report saved successfully.")


MISCLASSIFICATION AND ROOT-CAUSE ANALYSIS
Heart Disease Classification

1. OBJECTIVE

To examine false positives and false negatives, cluster similar error
cases, detect unusual feature values, and investigate subgroup error rates.

2. MODEL RESULTS

Test records: 61
Misclassified records: 13
False positives: 10
False negatives: 3
Overall error rate: 0.2131
Accuracy: 0.7869

3. ERROR CLUSTERING

Cluster counts: {0: 7, 2: 5, 1: 1}

Clusters represent groups of records with similar standardized
feature patterns. They do not represent clinical diagnoses.

4. ANOMALY ANALYSIS

Potential anomalies were flagged using the IQR rule.
Unusual values require verification and should not automatically
be treated as data errors.

5. SUBGROUP ANALYSIS

  Feature Group  Records  Errors  Error Rate
       cp     3        8       3    0.375000
       cp     2       17       5    0.294118
      sex     1       41      12    0.292683
Age Group   60+       18       5    0.277778
Age Group 50-59       27  

In [10]:
fp_count = int(((y_test == 0) & (predictions == 1)).sum())

fn_count = int(((y_test == 1) & (predictions == 0)).sum())